In [1]:
import openml
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
datasets = openml.datasets.list_datasets(output_format="dataframe")
datasets[datasets["name"].str.contains("telco", case=False, na=False)][["did","name","NumberOfInstances","NumberOfFeatures"]].head(20)

,did,name,NumberOfInstances,NumberOfFeatures
42178,42178,telco-customer-churn,7043.0,20.0
44228,44228,Churn_Telco_Europa,190776.0,20.0
44229,44229,Churn_Telco_Europa,1401.0,16.0
45568,45568,telco-customer-churn,7043.0,20.0


In [2]:
# %pip install openml

Загружаем данные

In [3]:
did = 42178
dataset = openml.datasets.get_dataset(did)

X, y, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute 
)

df = X.copy()
df["Churn"] = y

In [ ]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


: 

In [ ]:
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).str.strip()

if "TotalCharges" in df.columns:
    df["TotalCharges"] = df["TotalCharges"].replace("", np.nan)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

y = df["Churn"].map({"No": 0, "Yes": 1}).astype(int).values
X = df.drop(columns=["Churn"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_cols = X_train.select_dtypes(include=["int64","float64","uint8","int32","float32"]).columns
cat_cols = X_train.select_dtypes(include=["object","category","bool"]).columns

gender              0.0
SeniorCitizen       0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64
gender               object
SeniorCitizen         uint8
Partner              object
Dependents           object
tenure                uint8
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
target               object
dtype: object


Dataset + модель + train/eval

In [ ]:
class NumpyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.int64))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)  # 2 класса
        )

    def forward(self, x):
        return self.net(x)


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_probs, all_y = [], []
    correct, total = 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)[:, 1]
        pred = logits.argmax(dim=1)

        correct += (pred == yb).sum().item()
        total += yb.size(0)

        all_probs.append(probs.cpu().numpy())
        all_y.append(yb.cpu().numpy())

    acc = correct / total
    return acc, np.concatenate(all_probs), np.concatenate(all_y)


def train_one_run(Xtr, ytr, Xte, yte, seed=42, epochs=20, batch_size=64, lr=1e-3):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader = DataLoader(NumpyDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(NumpyDataset(Xte, yte), batch_size=batch_size, shuffle=False)

    model = MLP(Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

    acc, proba, y_true = eval_model(model, test_loader, device)
    auc = roc_auc_score(y_true, proba)
    return acc, auc

In [ ]:
def build_preprocess(num_impute: str, cat_impute: str, scaler: str):
    # числовая импутация
    if num_impute == "mean":
        num_imputer = SimpleImputer(strategy="mean")
    elif num_impute == "median":
        num_imputer = SimpleImputer(strategy="median")
    elif num_impute == "constant0":
        num_imputer = SimpleImputer(strategy="constant", fill_value=0)
    else:
        raise ValueError("num_impute must be mean/median/constant0")

    # масштабирование
    if scaler == "standard":
        scaler_obj = StandardScaler()
    elif scaler == "robust":
        scaler_obj = RobustScaler()
    elif scaler == "minmax":
        scaler_obj = MinMaxScaler()
    else:
        raise ValueError("scaler must be standard/robust/minmax")

    num_pipe = Pipeline([
        ("imputer", num_imputer),
        ("scaler", scaler_obj),
    ])

    # категориальная импутация
    if cat_impute == "most_frequent":
        cat_imputer = SimpleImputer(strategy="most_frequent")
    elif cat_impute == "missing":
        cat_imputer = SimpleImputer(strategy="constant", fill_value="missing")
    else:
        raise ValueError("cat_impute must be most_frequent/missing")

    cat_pipe = Pipeline([
        ("imputer", cat_imputer),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocess = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")

    return preprocess

In [ ]:
configs = [
    ("median", "most_frequent", "standard"),
    ("mean", "most_frequent", "standard"),
    ("constant0", "missing", "standard"),
    ("median", "most_frequent", "robust"),
    ("mean", "most_frequent", "robust"),
    ("constant0", "missing", "robust"),
    ("median", "most_frequent", "minmax"),
]

seeds = [42, 7, 123, 999, 2024]
rows = []

for num_imp, cat_imp, scaler in configs:
    preprocess = build_preprocess(num_imp, cat_imp, scaler)

    # fit на train, transform на test
    Xtr_p = preprocess.fit_transform(X_train)
    Xte_p = preprocess.transform(X_test)

    accs, aucs = [], []
    for s in seeds:
        acc, auc = train_one_run(Xtr_p, y_train, Xte_p, y_test, seed=s, epochs=20, batch_size=64, lr=1e-3)
        accs.append(acc)
        aucs.append(auc)

    rows.append({
        "num_impute": num_imp,
        "cat_impute": cat_imp,
        "scaler": scaler,
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs, ddof=1)),
        "auc_mean": float(np.mean(aucs)),
        "auc_std": float(np.std(aucs, ddof=1)),
        "n_seeds": len(seeds),
        "n_features_after": int(Xtr_p.shape[1]),
    })

results_df = pd.DataFrame(rows).sort_values("auc_mean", ascending=False)
results_df